<a href="https://colab.research.google.com/github/karthikkodali/Predictive-Analysis-Lab/blob/Credit-card/CreditCard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --- 1. Load and Explore the Data ---

# Load the dataset from the CSV file
file_name = list(uploaded.keys())[0]
data = pd.read_csv(io.BytesIO(uploaded[file_name]), on_bad_lines='skip')
#data = pd.read_csv('/content/dataset.csv', on_bad_lines='skip') # Added on_bad_lines='skip'


print("--- Data Head ---")
print(data.head())
print("\n--- Data Description ---")
print(data.describe())

# Check for the class imbalance
print("\n--- Class Distribution ---")
# Check if 'Class' column exists before accessing it
if 'Class' in data.columns:
    class_distribution = data['Class'].value_counts()
    print(class_distribution)
    # Check if both classes exist before printing counts
    if 0 in class_distribution:
        print(f"\nLegitimate Transactions (Class 0): {class_distribution[0]}")
    if 1 in class_distribution:
        print(f"Fraudulent Transactions (Class 1): {class_distribution[1]}")
        print(f"Percentage of Fraud: {class_distribution[1] / len(data) * 100:.4f}%")
    print("-" * 30)
else:
    print(" 'Class' column not found in the dataset. This dataset may not be suitable for fraud detection.")


# --- 2. Pre-processing ---

# The 'Time' and 'Amount' columns are not scaled like the others (V1, V2, etc.).
# We'll scale them to prevent them from overly influencing the model.
# Removed scaling for 'Time' and 'Amount' as they are not in the dataset and not relevant for this analysis.
# scaler = StandardScaler()
# data['scaled_Amount'] = scaler.fit_transform(data['Amount'].values.reshape(-1, 1))
# We can drop the original 'Time' and 'Amount' columns
# data = data.drop(['Time', 'Amount'], axis=1)

# Drop rows with missing values in the 'Class' column
if 'Class' in data.columns:
  data.dropna(subset=['Class'], inplace=True)
else:
  print("'Class' column not found, skipping dropping rows with missing values in 'Class'.")


# --- 3. Prepare Data for Modeling ---

# Define features (X) and target (y)
if 'Class' in data.columns:
  X = data.drop('Class', axis=1)
  y = data['Class']
else:
  print("'Class' column not found. Please define a target variable for modeling.")
  # Exit or handle the absence of a target variable appropriately
  # For now, I will set X and y to None to prevent further errors
  X = None
  y = None


# Split the data into training and testing sets
# We use 'stratify=y' to ensure the class distribution is the same in train and test sets
if X is not None and y is not None:
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42, stratify=y
  )

  # --- 4. Train a Baseline Model (Logistic Regression) ---
  print("\n--- Training Logistic Regression (Baseline) ---")
  lr_model = LogisticRegression(random_state=42)
  lr_model.fit(X_train, y_train)

  # Make predictions
  y_pred_lr = lr_model.predict(X_test)

  print("\n--- Logistic Regression Results ---")
  print("Confusion Matrix:")
  # Note: In a confusion matrix, the rows are the actual classes and columns are the predicted classes.
  # [[True Negatives, False Positives],
  #  [False Negatives, True Positives]]
  print(confusion_matrix(y_test, y_pred_lr))
  print("\nClassification Report:")
  print(classification_report(y_test, y_pred_lr, target_names=['Not Fraud (0)', 'Fraud (1)']))


  # --- 5. Train an Advanced Model (Random Forest) ---
  # Random Forest is better for complex, non-linear problems and imbalanced data.
  # `class_weight='balanced'` tells the model to pay more attention to the minority class (fraud).
  print("\n--- Training Random Forest Classifier ---")
  rf_model = RandomForestClassifier(
      n_estimators=100,
      random_state=42,
      class_weight='balanced',
      n_jobs=-1  # Use all available CPU cores
  )
  rf_model.fit(X_train, y_train)

  # Make predictions
  y_pred_rf = rf_model.predict(X_test)

  print("\n--- Random Forest Results ---")
  print("Confusion Matrix:")
  print(confusion_matrix(y_test, y_pred_rf))
  print("\nClassification Report:")
  print(classification_report(y_test, y_pred_rf, target_names=['Not Fraud (0)', 'Fraud (1)']))


  # --- 6. Summary and Interpretation ---
  print("\n--- Model Comparison Summary ---")
  # For fraud (class 1):
  lr_recall = confusion_matrix(y_test, y_pred_lr)[1, 1] / (confusion_matrix(y_test, y_pred_lr)[1, 1] + confusion_matrix(y_test, y_pred_lr)[1, 0])
  rf_recall = confusion_matrix(y_test, y_pred_rf)[1, 1] / (confusion_matrix(y_test, y_pred_rf)[1, 1] + confusion_matrix(y_test, y_pred_rf)[1, 0])

  print(f"Logistic Regression caught {lr_recall*100:.2f}% of the fraud cases in the test set.")
  print(f"Random Forest caught {rf_recall*100:.2f}% of the fraud cases in the test set.")

  print("\nInterpretation:")
  print("The Logistic Regression model has high precision but very poor recall for fraud cases. It correctly identified only a portion of the fraudulent transactions.")
  print("The Random Forest model, especially with `class_weight='balanced'`, performs much better. Its recall is significantly higher, meaning it successfully identified a much larger percentage of the actual fraud cases, even if it meant incorrectly flagging a few more legitimate transactions (lower precision).")
  print("In fraud detection, high recall is often the primary goal.")
else:
  print("\nSkipping model training and evaluation due to missing target variable.")

--- Data Head ---
  Product_ID        Date  Base_Price  Base_Price.1  Discount_Offered  \
0      P1023  15-01-2024         350           351                15   
1      P1045  20-02-2024         280           281                10   
2      P1078  05-03-2024         420           421                20   
3      P1099  18-04-2024         150           151                 5   
4      P1122  22-05-2024         390           391                25   

   Units_Sold  Revenue  Marketing_Spend  Customer_Rating  Inventory_Available  \
0         120      NaN             3200              4.2                  200   
1          95      NaN             1800              3.8                  150   
2         180      NaN             4500              4.5                  300   
3          75      NaN             1200              3.5                  100   
4         210      NaN             4800              4.7                  250   

  Clicked_Ad Customer_Segment       Event  Season  
0        Y